# 04 · Logistics — Trucks & Harvest
Explore truck fleet routes, harvest records, and probe the logistics endpoints
end-to-end.

**Covered endpoints**
- `GET  /trucks` — list trucks
- `GET  /trucks/{id}` — single truck detail
- `POST /trucks` — register a truck
- `POST /trucks/{id}/route` — assign route
- `GET  /harvest` — harvest records
- `POST /harvest` — record a harvest event
- `GET  /harvest/{id}` — single harvest detail

In [ ]:
import json, urllib.request, urllib.error
API_BASE = 'https://api.banana.engineer'

def pp(obj):
    if isinstance(obj, str):
        try: obj = json.loads(obj)
        except Exception: print(obj); return
    print(json.dumps(obj, indent=2))

def call_endpoint(method, path, payload=None, headers=None):
    url = f"{API_BASE}{path}"
    h = {'Accept': 'application/json'}
    if headers: h.update(headers)
    data = None
    if payload is not None:
        h['Content-Type'] = 'application/json'
        data = json.dumps(payload).encode()
    req = urllib.request.Request(url, data=data, headers=h, method=method.upper())
    try:
        with urllib.request.urlopen(req) as r:
            return {'ok': True, 'status': r.status, 'body': r.read().decode('utf-8', errors='replace')}
    except urllib.error.HTTPError as e:
        return {'ok': False, 'status': e.code, 'body': e.read().decode('utf-8', errors='replace')}
    except Exception as ex:
        return {'ok': False, 'status': None, 'body': str(ex)}

def get(path, **kw):  return call_endpoint('GET',  path, **kw)
def post(path, **kw): return call_endpoint('POST', path, **kw)
print('setup_ok')

In [ ]:
# ── 1. List trucks ────────────────────────────────────────────────────────
r = get('/trucks')
print('status:', r['status'])
pp(r['body'])

# Extract truck IDs for downstream probes
try:
    trucks = json.loads(r['body'])
    truck_ids = [t.get('id') or t.get('truckId') for t in (trucks if isinstance(trucks, list) else [])]
    print('truck_ids:', truck_ids)
except Exception:
    truck_ids = []

In [ ]:
# ── 2. Single truck detail (first from list) ──────────────────────────────
if truck_ids and truck_ids[0]:
    r = get(f'/trucks/{truck_ids[0]}')
    print('status:', r['status'])
    pp(r['body'])
else:
    print('no_truck_ids_available — run cell 1 first or adjust manually')

In [ ]:
# ── 3. List harvest records ───────────────────────────────────────────────
r = get('/harvest')
print('status:', r['status'])
pp(r['body'])

try:
    harvests = json.loads(r['body'])
    harvest_ids = [h.get('id') or h.get('harvestId') for h in (harvests if isinstance(harvests, list) else [])]
    print('harvest_ids:', harvest_ids)
except Exception:
    harvest_ids = []

In [ ]:
# ── 4. Single harvest detail ──────────────────────────────────────────────
if harvest_ids and harvest_ids[0]:
    r = get(f'/harvest/{harvest_ids[0]}')
    print('status:', r['status'])
    pp(r['body'])
else:
    print('no_harvest_ids_available')

In [ ]:
# ── 5. Route availability matrix ─────────────────────────────────────────
# For each truck, probe the route endpoint and collect availability
matrix = []
for tid in truck_ids[:5]:  # cap at 5 to avoid rate limits
    r = get(f'/trucks/{tid}')
    matrix.append({'truck_id': tid, 'status': r['status'], 'ok': r['ok']})

print('route_matrix:')
for row in matrix:
    print(row)